# Lesson 04 - ਟੂਲ ਯੂਜ਼ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕ੍ਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ (ਪਾਇਥਨ) ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹੋਏ AI ਏਜੰਟਾਂ ਲਈ **ਟੂਲ ਯੂਜ਼** ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਸਿੱਖੋਗੇ। ਅਸੀਂ ਇਹਨਾਂ ਜ਼ਰੀਏ ਕਵਰ ਕਰਾਂਗੇ:

- `@tool` ਡਿਕੋਰੇਟਰ ਅਤੇ ਟਾਈਪ ਕੀਤੇ ਪੈਰਾਮੀਟਰਾਂ ਦੇ ਨਾਲ ਫੰਕਸ਼ਨ ਟੂਲ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ
- ਟੂਲ ਸਕੀਮਾਵਾਂ ਪ੍ਰਦਾਨ ਕਰਨਾ ਤਾਂ ਜੋ ਮਾਡਲ ਨੂੰ ਪਤਾ ਹੋਵੇ ਕਿ ਹਰ ਟੂਲ ਕੀ ਕਰਦਾ ਹੈ
- `approval_mode` ਨਾਲ ਟੂਲ ਐਕਜ਼ੀਕਿਊਸ਼ਨ ਨੂੰ ਕੰਟਰੋਲ ਕਰਨਾ
- ਪਾਇਡੈਂਟਿਕ ਮਾਡਲਾਂ ਅਤੇ `response_format` ਰਾਹੀਂ **ਸੰਰਚਿਤ ਆਉਟਪੁਟ** ਵਾਪਸ ਕਰਨਾ

ਸਿਰੀਓ ਇੱਕ **ਟ੍ਰੈਵਲ ਬੁਕਿੰਗ ਏਜੰਟ** ਹੈ ਜੋ ਮੁਕਾਮਾਂ ਨੂੰ ਲੱਭ ਸਕਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਜਾਂਚ ਸਕਦਾ ਹੈ ਅਤੇ ਉਡਾਣ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰ ਸਕਦਾ ਹੈ।


## ਸੈੱਟਅਪ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ਡੈਕੋਰੇਟਰ ਨਾਲ ਟੂਲ ਪਰਿਭਾਸ਼ਤ ਕਰਨਾ

`@tool` ਡੈਕੋਰੇਟਰ ਇੱਕ ਸਧਾਰਣ ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਨੂੰ ਇੱਕ ਟੂਲ ਵਿੱਚ ਬਦਲ ਦਿੰਦਾ ਹੈ ਜਿਸਨੂੰ ਇੱਕ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।  
ਮੁੱਖ ਬਿੰਦੂ:

- **ਡੌਕਸਟ੍ਰਿੰਗ** ਮਾਡਲ ਵੱਲੋਂ ਵੇਖੀ ਜਾਣ ਵਾਲੀ ਟੂਲ ਵਰਣਨਾ ਬਣ ਜਾਂਦੀ ਹੈ।  
- **ਟਾਈਪ ਐਨੋਟੇਸ਼ਨਾਂ** (ਵਰਣਨ ਸਮੇਤ `Annotated` ਸਮੇਤ) ਟੂਲ ਸਕੀਮਾ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਨ।  
- `approval_mode` ਇਹ ਨਿਰਧਾਰਤ ਕਰਦਾ ਹੈ ਕਿ ਯੂਜ਼ਰ ਨੂੰ ਹਰ ਕਾਲ ਅਧੀਨ ਮਨਜ਼ੂਰੀ ਦੇਣੀ ਲਾਜ਼ਮੀ ਹੈ ਜਾਂ ਨਹੀਂ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ਕਈ ਸੰਦਾਂ ਨਾਲ ਏਜੰਟ ਬਣਾਉਣਾ

ਤਿੰਨੋ ਸੰਦਾਂ ਨੂੰ ਕਲਾਇੰਟ ਨੂੰ ਦਿੱਤੋ ਤਾਂ ਜੋ ਮਾਡਲ ਉਹਨਾਂ ਵਿੱਚੋਂ ਜੋ ਵੀ ਚਾਹੀਦਾ ਹੋਵੇ, ਉਪਭੋਗਤਾ ਦੇ ਸਵਾਲ ਦਾ ਜਵਾਬ ਦੇਣ ਲਈ ਬੁਲਾ ਸਕੇ।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output with Tools

`response_format` ਨੂੰ ਇੱਕ Pydantic ਮਾਡਲ 'ਤੇ ਸੈੱਟ ਕਰਨ ਨਾਲ, ਏਜੰਟ ਨੂੰ ਮੁਫ਼ਤ-ਰੂਪ ਟੈਕਸਟ ਦੀ ਥਾਂ ਇੱਕ ਚੰਗੇ ਤਰੀਕੇ ਨਾਲ ਪ੍ਰਕਾਰਿਤ JSON ਔਬਜੈਕਟ ਵਾਪਸ ਕਰਨ ਲਈ ਮਜਬੂਰ ਕੀਤਾ ਜਾਂਦਾ ਹੈ। ਜਦੋਂ ਨੀਚਲੇ ਕੋਡ ਨੂੰ ਪ੍ਰੋਗਰਾਮੈਟਿਕ ਤਰੀਕੇ ਨਾਲ ਨਤੀਜੇ ਦੀ ਵਰਤੋਂ ਕਰਨ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ ਤਾਂ ਇਹ ਲਾਭਦਾਇਕ ਹੁੰਦਾ ਹੈ।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ਸੰਦ ਮਨਜ਼ੂਰੀ ਪੈਟਰਨ

`@tool` 'ਤੇ `approval_mode` ਪੈਰਾਮੀਟਰ ਨਿਰਧਾਰਤ ਕਰਦਾ ਹੈ ਕਿ ਸੰਦ ਕਾਲਾਂ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਮਨੁੱਖੀ ਮਨਜ਼ੂਰੀ ਦੀ ਲੋੜ ਹੈ ਜਾਂ ਨਹੀਂ:

| ਮੋਡ | ਵਿਹਾਰ |
|---|---|
| `"never_require"` | ਸੰਦ ਆਟੋਮੈਟਿਕ ਰੂਪ ਵਿੱਚ ਚਲਦਾ ਹੈ — ਕਿਸੇ ਉਪਭੋਗਤਾ ਦੀ ਪੁਸ਼ਟੀ ਦੀ ਲੋੜ ਨਹੀਂ। |
| `"always_require"` | ਹਰ ਕਾਲ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਉਪਭੋਗਤਾ ਵੱਲੋਂ ਮਨਜ਼ੂਰੀ ਲੈਣੀ ਲਾਜ਼ਮੀ ਹੈ। |

ਜਿਹੜੇ ਸੰਦਾਂ ਦੇ ਸਾਈਡ-ਪ੍ਰਭਾਵ ਹੁੰਦੇ ਹਨ (ਜਿਵੇਂ ਕਿ ਫਲਾਈਟ ਬੁਕਿੰਗ, ਕਰੈਡਿਟ ਕਾਰਡ ਚਾਰਜ ਕਰਨਾ) ਲਈ `"always_require"` ਦੀ ਵਰਤੋਂ ਕਰੋ ਤਾਂ ਜੋ ਮਨੁੱਖ ਅੰਦਰ ਲੂਪ ਵਿਚ ਰਹਿ ਸਕੇ।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ:

1. **ਟੂਲ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨਾ** `@tool` ਡੇਕੋਰੇਟਰ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਟਾਈਪ ਕੀਤੀਆਂ ਪੈਰਾਮੀਟਰਾਂ ਅਤੇ ਡੌਕਸਟ੍ਰਿੰਗਜ਼ ਨਾਲ ਜੋ ਟੂਲ ਸਕੀਮਾ ਵਜੋਂ ਕੰਮ ਕਰਦੀਆਂ ਹਨ।
2. **ਕਈ ਟੂਲਾਂ ਨੂੰ ਜੋੜਨਾ** ਤਾਂ ਜੋ ਏਜੰਟ ਉਨ੍ਹਾਂ ਨੂੰ ਲੜੀਵਾਰ ਕਾਲ ਕਰਕੇ ਜਟਿਲ ਪ੍ਰਸ਼ਨਾਂ ਦੇ ਜਵਾਬ ਦੇ ਸਕੇ।
3. **ਸੰਰਚਿਤ ਨਤੀਜਾ ਵਾਪਸ ਕਰਨਾ** ਇੱਕ ਪਾਈਡੈਂਟਿਕ ਮਾਡਲ ਨੂੰ `response_format` ਵਜੋਂ ਪਾਸ ਕਰਕੇ।
4. **ਟੂਲ ਮਨਜ਼ੂਰੀ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰਨਾ** `approval_mode` ਨਾਲ ਤਾਂ ਜੋ ਸੰਵੇਦਨਸ਼ੀਲ ਕਾਰਵਾਈਆਂ ਲਈ ਮਨੁੱਖੀ ਹਸਤਕਸ਼ੇਪ ਜਾਰੀ ਰਹੇ।

ਇਹ ਪੈਟਰਨ ਭਰੋਸੇਮੰਦ, ਉਤਪਾਦਨ-ਤਿਆਰ ਏਜੰਟ ਬਣਾਉਣ ਦਾ ਆਧਾਰ ਬਣਾਉਂਦੇ ਹਨ ਜੋ ਬਾਹਰੀ ਪ੍ਰਣਾਲੀਆਂ ਨਾਲ ਸੁਰੱਖਿਅਤ ਤਰੀਕੇ ਨਾਲ ਇੰਟਰੈਕਟ ਕਰ ਸਕਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰਤਾ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸੰਗਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਉੱਭਰ ਰਹੀਆਂ ਕਿਸੇ ਵੀ ਸਮਝਦਾਰੀ ਜਾਂ ਭੁਲ-ਫਹਿਮੀ ਲਈ ਅਸੀਂ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
